## **DATA READING**

#### Data Reading JSON

In [0]:
dbutils.fs.ls('/Volumes/workspace/tutorial/myvolume/')

In [0]:
df_json = spark.read.format('json').option('inferSchema', True)\
            .option('header',True)\
            .option('multiLine', False)\
            .load('/Volumes/workspace/tutorial/myvolume/drivers.json')
    

In [0]:
df_json.display()

#### Data Reading


In [0]:
# Read csv file into dataframe
df = spark.read.format('csv').option('inferSchema',True).option('header',True).load('/Volumes/workspace/tutorial/myvolume/BigMartSales.csv')

In [0]:
df.display()

In [0]:
df.printSchema()

#### DDL Schema

In [0]:
# Change Item_Weight type to string (my schema, my rules)
my_ddl_schema = '''
    Item_Identifier string,
    Item_Weight string, 
    Item_Fat_Content string,
    Item_Visibility double,
    Item_Type string,
    Item_MRP double,
    Outlet_Identifier string,
    Outlet_Establishment_Year integer,
    Outlet_Size string,
    Outlet_Location_Type string,
    Outlet_Type string,
    Item_Outlet_Sales double
'''

In [0]:
df = spark.read.format('csv').schema(my_ddl_schema).option('header',True).load('/Volumes/workspace/tutorial/myvolume/BigMartSales.csv')


In [0]:
df.display()

#### StructType() Schema

In [0]:
from pyspark.sql.types  import *
from pyspark.sql.functions import *

In [0]:
# StructField
my_strct_schema = StructType([
    StructField('Item_Identifier',StringType(),True),
    StructField('Item_Weight',StringType(),True),
    StructField('Item_Fat_Content',StringType(),True),
    StructField('Item_Visibility',StringType(),True),
    StructField('Item_MRP',StringType(),True),
    StructField('Outlet_Identifier',StringType(),True),
    StructField('Outlet_Establishment_Year',StringType(),True),
    StructField('Outlet_Size',StringType(),True),
    StructField('Outlet_Location_Type',StringType(),True),
    StructField('Outlet_Type',StringType(),True),
    StructField('Item_Outlet_Sales',StringType(),True)
])

In [0]:
df = spark.read.format('csv').schema(my_strct_schema).option('header',True).load('/Volumes/workspace/tutorial/myvolume/BigMartSales.csv')

In [0]:
df.printSchema()

## **TRANSFORMATIONS**

#### SELECT

In [0]:
df.select(col("Item_Identifier"),col("Item_Weight"), col("Item_Fat_Content")).display()

In [0]:
df.select(col("Item_Identifier").alias('Item_ID')).display()

In [0]:
df.display()

#### FILTER

In [0]:
# Scenario 1
df.filter(col("Item_Fat_Content")=='Regular').display()

In [0]:
#Scenario 2
df.filter((col('Item_Type') == 'Soft Drinks') & (col('Item_Weight')<10)).display()  
df.filter((col('Item_Type') == 'Soft Drinks') & (col('Item_Weight')<10)).display()
# NameError: name 'col' is not defined
# import * from pyspark.sql.types and .functions

In [0]:
# Scenario 3
df.filter((col('Outlet_Size').isNull()) & (col("Outlet_Location_Type").isin('Tier 1', 'Tier 2'))).display()

In [0]:
# Rename Column
df.withColumnRenamed('Item_Weight', 'Item_wt').display()

In [0]:
# Create new column with value for each column as "new"

df = df.withColumn('flag',lit("new"))

In [0]:
df.display()

In [0]:
# withColumn
df.withColumn('Multiply',col('Item_Weight')*col("Item_MRP")).display()

In [0]:
df.withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),"Regular","Reg"))\
    .withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),"Low Fat","Lf")).display()

In [0]:
# Type Casting
df = df.withColumn('Item_Weight',col('Item_Weight').cast(StringType()))

#### SORT

In [0]:
# Sort / Order By

df.sort(col("Item_Weight").desc()).display()

In [0]:
df.sort(["Item_Weight", "Item_Visibility"], ascending=[0, 1]).display()

In [0]:
# Limit

df.limit(10).display()

#### DROP

In [0]:
df.drop('Item_Visibility','Item_Type').display()

In [0]:
# Drop Duplicates

df.dropDuplicates(subset=['Item_Type']).display()

In [0]:
# Distinct does the same thing as Drop Duplicates without subsets (for all columns, unless chosen)

df.distinct().display()

#### UNION and UNION BY NAME

In [0]:
data1 = [('1','kad'),
        ('2','sid')]
schema1 = 'id STRING, name STRING' 

df1 = spark.createDataFrame(data1,schema1)

data2 = [('3','rahul'),
        ('4','jas')]
schema2 = 'id STRING, name STRING' 

df2 = spark.createDataFrame(data2,schema2)

In [0]:
df1.display() 
df2.display()

In [0]:
# Union
df1.union(df2).display()

In [0]:
# Union by name
df1.unionByName(df2).display()

#### STRING FUNCTIONS

In [0]:
# Initcap
df.select(initcap('Item_Type')).dropDuplicates().display()

In [0]:
# upper() or lower() 
df.select(upper('Item_Type').alias('Upper Item Type')).display()

#### DATE FUNCTIONS

In [0]:
#CURRENT_DATE()

df = df.withColumn('curr_date',current_date())

df.display()

In [0]:
df = df.withColumn('Week_after',date_add(df.curr_date,7))

df.display()

In [0]:
df = df.withColumn('datediff',datediff('week_after','curr_date'))
df = df.withColumn('week_before', date_add('curr_date', -7))
df.select('curr_date', 'Week_after', 'week_before', 'datediff').limit(10).display()

#### NULL HANDLING 

#### Dropping Nulls

In [0]:
df.dropna(subset=["Outlet_Size"]).display()

#### Filling Nulls

In [0]:
df.fillna('Not Available', subset = ["Item_Weight"]).limit(100).display()

#### SPLIT and Indexing

##### SPLIT

In [0]:
df.withColumn('Outlet_Type',split('Outlet_Type',' ')).display()

##### Indexing

In [0]:
df.withColumn('Outlet_Type',split('Outlet_Type',' ')[1]).display()

##### Explode

In [0]:
# Creating new dataFrame
df_exp = df.withColumn('Outlet_Type',split('Outlet_Type',' '))
df_exp.limit(25).display()

In [0]:
# Using explode function
df_exp.withColumn('Outlet_Type',explode('Outlet_Type')).display()

In [0]:
df_exp.limit(10).display()

##### ARRAY_CONTAINS

In [0]:
df_exp.withColumn('Type1_flag',array_contains('Outlet_Type','Type1')).limit(50).display()

#### GROUP BY

##### Scenario 1

In [0]:
df.groupBy("Item_Type").agg(sum("Item_MRP")).limit(50).display()

##### Scenario 2

In [0]:
#Average
df.groupBy("Item_Type").agg(avg("Item_MRP")).limit(50).display()

##### Scenario 3

In [0]:
# Sum of MRP grouped by Item_Type & Outlet_Size
df.groupBy("Item_Type", "Outlet_Type").agg(sum("Item_MRP")).alias('Total_MRP').orderBy('Item_Type').display()

##### Scenario 4

In [0]:
## GroupBy on both columns, then Sum & Avg MRP

df.groupBy("Item_Type", "Outlet_Type").agg(sum("Item_MRP"),avg("Item_MRP")).display()

#### COLLECT_LIST - Like group_concat in SQL

In [0]:
data = [('user1','book1'),
        ('user1','book2'),
        ('user2','book2'),
        ('user2','book4'),
        ('user3','book1')]

schema = 'user string, book string' # can work without schema as well

# Create empty DataFrame by passing ' ' as first arg
df_book = spark.createDataFrame(data,schema)

df_book.display()

In [0]:
df_book.groupBy('user').agg(collect_list('book')).display()

#### PIVOT

In [0]:
# Pivot on Outlet Size
df.groupBy('Item_Type').pivot('Outlet_Size').agg(sum('Item_MRP')).display()

#### WHEN-OTHERWISE

In [0]:
df.display()

##### Scenario 1

In [0]:
df = df.withColumn('veg_flag',when(col("Item_Type")=='Meat','Non-Veg').otherwise('Veg'))

##### Scenario 2

In [0]:
df.withColumn('veg_exp_flag', when(((col("veg_flag")=='Veg') & (col("Item_MRP")<100)), 'Veg_Inexpensive')\
        .when((col("veg_flag")=='Veg') & (col("Item_MRP")<100), 'Veg_Expensive')\
        .otherwise('Non-Veg')).display()

#### JOINS

In [0]:
# DataFrames for Join Practice
dataj1 = [('1','gaur','d01'),
          ('2','kit','d02'),
          ('3','sam','d03'),
          ('4','tim','d03'),
          ('5','aman','d05'),
          ('6','nad','d06')] 

schemaj1 = 'emp_id STRING, emp_name STRING, dept_id STRING' 

df1 = spark.createDataFrame(dataj1,schemaj1)

dataj2 = [('d01','HR'),
          ('d02','Marketing'),
          ('d03','Accounts'),
          ('d04','IT'),
          ('d05','Finance')]

schemaj2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(dataj2,schemaj2)

In [0]:
df1.display()
df2.display()

##### Inner Join

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'], "inner").display()

##### Left join

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'], "left").fillna('Not Available').display()

##### Right Join

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'], "right").fillna('random_text').display()

##### Anti Join

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'], "anti").display()

#### WINDOW FUNCTIONS

In [0]:
from pyspark.sql import Window

##### ROW_NUMBER - Unique

In [0]:
df.withColumn('rowCol', row_number().over(Window.orderBy('Item_Identifier'))).display()

##### RANK and DENSE_RANK

In [0]:
df.withColumn('rowCol', row_number().over(Window.orderBy(col('Item_Identifier').desc())))\
    .withColumn('denseRank', dense_rank().over(Window.orderBy(col('Item_Identifier').desc())))\
    .withColumn('Rank', rank().over(Window.orderBy(col('Item_Identifier').desc()))).select("Item_Identifier","Rank","denseRank","rowCol").limit(100).orderBy('rowCol').display()

##### Cumulative Sum

In [0]:
# This will generate total sum in each row.
df.withColumn('cumulativeSum',sum('Item_MRP').over(Window.orderBy('Item_Type'))).display()

In [0]:
# rowsBetween(Window.unboundedPreceding, Window.currentRow)
df.withColumn('cumulativeSum',sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding, Window.currentRow))).display()

In [0]:
df.withColumn('cumulativeSum',sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing))).display()

#### USER DEFINED FUNCTIONS (UDF)

##### Step 1

In [0]:
def myfunc (x):
    return x * x

##### Step 2

In [0]:
# udf (function_name) converts Python function to an user defined function in PySpark

my_udf = udf(myfunc)

##### Step 3

In [0]:
df.withColumn('newColumnSquare',my_udf('item_MRP')).select('newColumnSquare','Item_MRP').display()

## **DATA WRITING**

#### CSV

In [0]:
df.write.format("csv")\
    .save('/Volumes/workspace/tutorial/myvolume/CSV/data.csv')

##### APPEND

In [0]:
df.write.format("csv")\
    .mode('append')\
    .save('/Volumes/workspace/tutorial/myvolume/CSV/data.csv')

In [0]:
df.write.format("csv")\
    .mode('append')\
    .option('path', '/Volumes/workspace/tutorial/myvolume/CSV/data.csv')\
    .save()

##### Overwrite

In [0]:
df.write.format("csv")\
    .mode('overwrite')\
    .option('path', '/Volumes/workspace/tutorial/myvolume/CSV/data.csv')\
    .save()

##### Error

In [0]:
df.write.format("csv")\
    .mode('error')\
    .option('path', '/Volumes/workspace/tutorial/myvolume/CSV/data.csv')\
    .save()

##### Ignore

In [0]:
df.write.format("csv")\
    .mode('ignore')\
    .option('path', '/Volumes/workspace/tutorial/myvolume/CSV/data.csv')\
    .save()

#### PARQUET

In [0]:
df.write.format("parquet")\
    .mode('overwrite')\
    .option('path', '/Volumes/workspace/tutorial/myvolume/CSV/data.csv')\
    .save()

#### TABLE

In [0]:
# Save as Table, .option is not needed, managed table
df.write.format('Delta')\
    .mode('overwrite')\
    .saveAsTable('my_table')

### SPARK SQL

#### createTempView 

In [0]:
# Create Temp View, and query Views like we query View in SQL - All Views are eliminated once the session ends.
df.createTempView('my_tempView')

# Create Or Replace
df.createOrReplaceTempView('my_view')

# Create Global Temporary View, Not supported on Serverless
# df.createOrReplaceGlobalTempView('my_globalView')

In [0]:
%sql

select * from my_view where Item_MRP < 100

In [0]:
# Save as DataFrame
df_sql = spark.sql("SELECT * from my_view WHERE Item_Fat_Content = 'Low Fat'")
df_sql.limit(100).display()